In [1]:
# install packages in colab if necessary
#!pip install sacremoses

import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.flow as nafc
import numpy as np
import pandas as pd
import torch
import gc
from tqdm import tqdm

tqdm.pandas()


rseed = 42

In [2]:
from huggingface_hub import login
from getpass import getpass

# Dies öffnet ein Eingabefeld, in das du deinen Token kopierst.
# Er wird dabei nicht im Klartext im Notebook gespeichert!
token = getpass("HF Token: ")
login(token=token)

HF Token:  ········


In [3]:
# models
# aug_syn_word = naw.SynonymAug(aug_src = "wordnet")

# aug_syn_bert =naw.ContextualWordEmbsAug(
#             model_path='bert-base-uncased',
#             action="substitute"
#         )

# aug_backtranslation = aug = naw.BackTranslationAug(
#         from_model_name="facebook/wmt19-en-de",
#         to_model_name="facebook/wmt19-de-en"
#     )

# aug_rand_swap = naw.RandomWordAug(action = "swap")
# aug_rand_del = naw.RandomWordAug(action = "delete")

In [4]:
# Synonym Augmenter
# prob 0.4
def synonymizer(text, aug1):
#    coin = np.random.rand()
#    if coin < 0.4:
    augmented_text = aug1.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text
#    else:
#        augmented_text = aug2.augment(text)
#        return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

In [5]:
# backtranslation
# prob 0.2
def back_translation(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

# random swap
# prob 0.2
def random_swap(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text


#  random deletion
# prob 0.2
def random_del(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text


In [6]:
# def augmenter(text, aug_syn, aug_bt, aug_sw, aug_del):
#     coin = np.random.rand()
#     if coin < 0.4:
#         return synonymizer(text, aug_syn)
#     elif 0.4 < coin < 0.6:
#         return back_translation(text, aug_bt)
#     elif 0.6 < coin < 0.8:
#         return random_swap(text, aug_sw)
#     else:
#         return random_del(text, aug_del)




In [7]:
# data local
#df = pd.read_csv("..\data\csv\mbti_cleaned.csv")

# data colab
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv("/content/drive/MyDrive/Data/mbti_cleaned.csv")

#data alvis
df = pd.read_csv("~/MA/data/mbti_cleaned.csv")

In [8]:
def label_samples(df, label, n_goal, rseed):
    obs_count = (df["label"] == label).sum()
    if obs_count < n_goal:
        df_subset = df[df["label"] == label]

        n_diff = n_goal - obs_count

        n_syn = round(n_diff*0.4)
        n_bt = round(n_diff*0.2)
        n_sw = round(n_diff*0.2)
        n_del = n_diff - (n_syn+n_bt+n_sw)



        syn_sample = df_subset.sample(n=n_syn, replace=True,random_state=rseed)
        bt_sample = df_subset.sample(n=n_bt, replace=True,random_state=rseed)
        sw_sample = df_subset.sample(n=n_sw, replace=True,random_state=rseed)
        del_sample = df_subset.sample(n=n_del, replace=True,random_state=rseed)

        return syn_sample, bt_sample, sw_sample, del_sample
    else:
        empty = df.iloc[:0]
        return empty, empty, empty, empty

    

    


In [9]:
syn_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
bt_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
sw_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
del_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])

for label in df["label"].unique():
    s1, s2, s3, s4 = label_samples(df, label, n_goal=10000, rseed=42)
    

    syn_sample = pd.concat([syn_sample, s1]).dropna()
    bt_sample = pd.concat([bt_sample, s2]).dropna()
    sw_sample = pd.concat([sw_sample, s3]).dropna()
    del_sample = pd.concat([del_sample, s4]).dropna()


    



In [10]:
def data_augmenter(syn_sample, bt_sample, sw_sample, del_sample):
    #device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    
    #aug_syn_word = naw.SynonymAug(aug_src = "wordnet")
    aug_syn_bert =naw.ContextualWordEmbsAug(
        model_path='bert-base-uncased',
        action="substitute",
        device="cuda"
    )

    syn_sample["post_augmented"] = syn_sample["post"].progress_apply(synonymizer, args=(aug_syn_bert,))

    #del aug_syn_word
    del aug_syn_bert

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    aug_backtranslation = naw.BackTranslationAug(
        from_model_name="facebook/wmt19-en-de",
        to_model_name="facebook/wmt19-de-en",
        device= "cuda",
        batch_size=16
    )

    texts = bt_sample["post"].tolist()
    augmented_texts = aug_backtranslation.augment(texts)
    bt_sample["post_augmented"] = augmented_texts

    del aug_backtranslation
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    aug_rand_swap = naw.RandomWordAug(action = "swap")
    sw_sample["post_augmented"] = sw_sample["post"].progress_apply(random_swap, args = (aug_rand_swap,))
    
    aug_rand_del = naw.RandomWordAug(action = "delete")
    del_sample["post_augmented"] = del_sample["post"].progress_apply(random_del, args = (aug_rand_del,))


    return syn_sample,bt_sample,sw_sample,del_sample



In [11]:
from transformers import BertTokenizer, AutoTokenizer

# Wir patchen die Klasse BertTokenizer, damit nlpaug die Methode findet
def patch_tokenizer():
    # Die neue Methode in transformers heißt convert_tokens_to_ids
    # nlpaug sucht aber nach _convert_token_to_id
    if not hasattr(BertTokenizer, '_convert_token_to_id'):
        BertTokenizer._convert_token_to_id = lambda self, token: self.convert_tokens_to_ids(token)
    print("Tokenizer-Patch angewendet!")

patch_tokenizer()

Tokenizer-Patch angewendet!


In [12]:
# run data_augmenter
syn_df, bt_df, sw_df, del_df = data_augmenter(syn_sample, bt_sample, sw_sample, del_sample)

The following layers were not sharded: cls.predictions.transform.LayerNorm.bias, bert.encoder.layer.*.output.dense.bias, bert.embeddings.position_embeddings.weight, bert.encoder.layer.*.attention.output.dense.bias, bert.encoder.layer.*.attention.self.value.weight, bert.encoder.layer.*.intermediate.dense.weight, bert.embeddings.token_type_embeddings.weight, bert.embeddings.LayerNorm.bias, bert.encoder.layer.*.attention.self.value.bias, bert.encoder.layer.*.output.dense.weight, bert.encoder.layer.*.attention.output.dense.weight, bert.encoder.layer.*.attention.output.LayerNorm.weight, bert.encoder.layer.*.attention.self.key.weight, cls.predictions.transform.dense.bias, bert.encoder.layer.*.intermediate.dense.bias, cls.predictions.decoder.bias, bert.encoder.layer.*.output.LayerNorm.weight, cls.predictions.bias, bert.encoder.layer.*.attention.self.query.bias, cls.predictions.transform.dense.weight, bert.embeddings.LayerNorm.weight, bert.encoder.layer.*.attention.self.query.weight, bert.enco

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

100%|██████████| 19162/19162 [19:29<00:00, 16.38it/s]


config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

vocab-src.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

100%|██████████| 9580/9580 [00:01<00:00, 7247.70it/s]


In [16]:
import pyarrow

syn_sample.to_csv('syn_augmented.csv', sep='\t', index=False, quoting=1)
bt_sample.to_csv('bt_augmented.csv', sep='\t', index=False, quoting=1)
sw_sample.to_csv('sw_augmented.csv', sep='\t', index=False, quoting=1)
del_sample.to_csv('del_augmented.csv', sep='\t', index=False, quoting=1)

print("Alle Dateien erfolgreich als CSV gespeichert!")

Alle Dateien erfolgreich als CSV gespeichert!
